# Exploration des datasets

In [59]:
import pandas as pd
import os

## Datasets

- yield.csv : contient les données de rendement agricole (hg/ha) par pays et par année.
- temp.csv : contient les données de température moyenne par pays et par année.
- rainfall.csv : contient les données de précipitations moyennes par pays et par année.
- pesticides.csv : contient les données de consommation de pesticides par pays et par année.
- yield_df.csv : contient les données de rendement agricole (hg/ha), pesticides utilisés, température moyenne et précipitation moyenne par pays et par année.
- crop_yield : contient les données de rendement agricole (hg/ha) pour différentes cultures selon différents paramètres.

## Fichier yield.csv

In [60]:
productivity = pd.read_csv('../data/raw/yield.csv', sep=',', na_values=[''], quotechar='"')

In [61]:
productivity.head(5)

,Domain Code,Domain,Area Code,Area,Element Code,Element,Item Code,Item,Year Code,Year,Unit,Value
0,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1961,1961,hg/ha,14000
1,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1962,1962,hg/ha,14000
2,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1963,1963,hg/ha,14260
3,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1964,1964,hg/ha,14257
4,QC,Crops,2,Afghanistan,5419,Yield,56,Maize,1965,1965,hg/ha,14400


In [62]:
productivity['Area'].nunique()

212

In [63]:
# Suppression des colonnes inutiles et renommage de la colonne Value en hg/ha
productivity.drop(columns=['Domain Code', 'Domain', 'Element Code', 'Element', 'Item Code', 'Year Code'], inplace=True)
productivity.rename(columns={'Value': 'hg/ha'}, inplace=True)
productivity.drop(columns=['Unit'], inplace=True)
productivity.head(5)

,Area Code,Area,Item,Year,hg/ha
0,2,Afghanistan,Maize,1961,14000
1,2,Afghanistan,Maize,1962,14000
2,2,Afghanistan,Maize,1963,14260
3,2,Afghanistan,Maize,1964,14257
4,2,Afghanistan,Maize,1965,14400


In [64]:
productivity['hg/ha'].describe()

count      56717.000000
mean       62094.660084
std        67835.932856
min            0.000000
25%        15680.000000
50%        36744.000000
75%        86213.000000
max      1000000.000000
Name: hg/ha, dtype: float64

In [65]:
productivity[productivity['hg/ha'] == 1000000]

,Area Code,Area,Item,Year,hg/ha
27930,114,Kenya,Plantains and others,1964,1000000


In [66]:
# retrait de l'outlier qui semble être une erreur de saisie
dataset = productivity[productivity['hg/ha'] != 1000000]

In [67]:
productivity[productivity['hg/ha'] == 0]

,Area Code,Area,Item,Year,hg/ha
35804,153,New Caledonia,Sorghum,2000,0
35806,153,New Caledonia,Sorghum,2012,0
35906,153,New Caledonia,Wheat,2000,0
35908,153,New Caledonia,Wheat,2002,0
35912,153,New Caledonia,Wheat,2006,0
35913,153,New Caledonia,Wheat,2007,0
35914,153,New Caledonia,Wheat,2011,0
37860,299,Occupied Palestinian Territory,Sorghum,1999,0


Une production à 0hg/ha n'est pas forcément un outlier et est peut-être indicatrice d'un événement ces années-la.

In [68]:
if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

productivity.to_csv('../data/processed/yield.csv', index=False)

## Fichier temp.csv

In [69]:
temp = pd.read_csv('../data/raw/temp.csv', sep=',', na_values=[''], quotechar='"')

In [70]:
temp.head(5)

,year,country,avg_temp
0,1849,Côte D'Ivoire,25.58
1,1850,Côte D'Ivoire,25.52
2,1851,Côte D'Ivoire,25.67
3,1852,Côte D'Ivoire,NaN
4,1853,Côte D'Ivoire,NaN


In [71]:
temp.duplicated().value_counts()

False    64353
True      6958
Name: count, dtype: int64

In [72]:
temp = temp.drop_duplicates()
temp.duplicated().value_counts()

False    64353
Name: count, dtype: int64

In [73]:
# Renommage des colonnes pour correspondre aux autres datasets
temp.rename(columns={'country': 'Area', 'year': 'Year'}, inplace=True)
temp.head(5)

,Year,Area,avg_temp
0,1849,Côte D'Ivoire,25.58
1,1850,Côte D'Ivoire,25.52
2,1851,Côte D'Ivoire,25.67
3,1852,Côte D'Ivoire,NaN
4,1853,Côte D'Ivoire,NaN


In [74]:
temp.duplicated(['Area', 'Year']).value_counts()

True     35839
False    28514
Name: count, dtype: int64

In [75]:
temp.info()

<class 'pandas.DataFrame'>
Index: 64353 entries, 0 to 71134
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Year      64353 non-null  int64  
 1   Area      64353 non-null  str    
 2   avg_temp  62976 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 2.5 MB


In [76]:
temp = temp.groupby(['Area','Year'], as_index=False)['avg_temp'].mean()

In [77]:
temp.info()

<class 'pandas.DataFrame'>
RangeIndex: 28514 entries, 0 to 28513
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Area      28514 non-null  str    
 1   Year      28514 non-null  int64  
 2   avg_temp  27479 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 899.9 KB


In [78]:
if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

temp.to_csv('../data/processed/temp.csv', index=False)

## Fichier rainfall.csv

In [79]:
rainfall = pd.read_csv('../data/raw/rainfall.csv', sep=',', na_values=[''], quotechar='"')
rainfall.info()

<class 'pandas.DataFrame'>
RangeIndex: 6727 entries, 0 to 6726
Data columns (total 3 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0    Area                          6727 non-null   str  
 1   Year                           6727 non-null   int64
 2   average_rain_fall_mm_per_year  5953 non-null   str  
dtypes: int64(1), str(2)
memory usage: 239.2 KB


In [80]:
rainfall.head(5)

,Area,Year,average_rain_fall_mm_per_year
0,Afghanistan,1985,327
1,Afghanistan,1986,327
2,Afghanistan,1987,327
3,Afghanistan,1989,327
4,Afghanistan,1990,327


On remarque que pour les précipitations, quelque soit l'année la valeur est la même. On peut donc remplacer les valeurs manquantes par la valeur de l'année 1990.
Mais il serait plus précis de remplacer cette valeur par des valeurs provenant d'un autre dataset. On va donc chercher un dataset contenant les précipitations par pays et par année.

In [81]:
# Dataset contenant les précipitations par pays et par année issues de https://ourworldindata.org/grapher/average-precipitation-per-year
average_precipitation_per_year = pd.read_csv('../data/raw/average-precipitation-per-year.csv', sep=',', na_values=[''], quotechar='"')
average_precipitation_per_year

,Entity,Code,Year,Annual precipitation
0,Afghanistan,AFG,1940,217.67816
1,Afghanistan,AFG,1941,196.80933
2,Afghanistan,AFG,1942,300.08908
3,Afghanistan,AFG,1943,345.60892
4,Afghanistan,AFG,1944,324.44350
...,...,...,...,...
16765,Zimbabwe,ZWE,2021,788.98170
16766,Zimbabwe,ZWE,2022,774.35420
16767,Zimbabwe,ZWE,2023,776.85040
16768,Zimbabwe,ZWE,2024,421.73400


In [82]:
average_precipitation_per_year.duplicated().value_counts()

False    16770
Name: count, dtype: int64

In [83]:
average_precipitation_per_year.rename(columns={'Entity': 'Area', 'Year': 'Year', 'Annual precipitation': 'average_rain_fall_mm_per_year'}, inplace=True)

In [84]:
average_precipitation_per_year.duplicated(['Area', 'Year']).value_counts()

False    16770
Name: count, dtype: int64

In [85]:
average_precipitation_per_year.info()

<class 'pandas.DataFrame'>
RangeIndex: 16770 entries, 0 to 16769
Data columns (total 4 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Area                           16770 non-null  str    
 1   Code                           16684 non-null  str    
 2   Year                           16770 non-null  int64  
 3   average_rain_fall_mm_per_year  16770 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 726.5 KB


In [86]:
if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

average_precipitation_per_year.to_csv('../data/processed/rainfall.csv', index=False)

## Fichier pesticides.csv

In [87]:
pesticides = pd.read_csv('../data/raw/pesticides.csv', sep=',', na_values=[''], quotechar='"')

In [88]:
pesticides.head(5)

,Domain,Area,Element,Item,Year,Unit,Value
0,Pesticides Use,Albania,Use,Pesticides (total),1990,tonnes of active ingredients,121.0
1,Pesticides Use,Albania,Use,Pesticides (total),1991,tonnes of active ingredients,121.0
2,Pesticides Use,Albania,Use,Pesticides (total),1992,tonnes of active ingredients,121.0
3,Pesticides Use,Albania,Use,Pesticides (total),1993,tonnes of active ingredients,121.0
4,Pesticides Use,Albania,Use,Pesticides (total),1994,tonnes of active ingredients,201.0


In [89]:
# Suppression des colonnes inutiles et renommage de la colonne Value en pesticides_tonnes
pesticides.drop(columns=['Domain', 'Element', 'Item', 'Unit'], inplace=True)
pesticides.rename(columns={'Value': 'pesticides_tonnes'}, inplace=True)
pesticides.head(5)

,Area,Year,pesticides_tonnes
0,Albania,1990,121.0
1,Albania,1991,121.0
2,Albania,1992,121.0
3,Albania,1993,121.0
4,Albania,1994,201.0


In [90]:
if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

pesticides.to_csv('../data/processed/pesticides.csv', index=False)

## Fichier yield_df.csv

In [91]:
yield_df = pd.read_csv('../data/raw/yield_df.csv', sep=',', na_values=[''], quotechar='"')

In [92]:
yield_df['Area'].nunique()

101

In [93]:
productivity['Area'].nunique()

212

In [94]:
import pandas as pd
yield_df = pd.read_csv('../data/raw/yield_df.csv', sep=',', na_values=[''], quotechar='"')

In [95]:
yc = yield_df['Area'].unique().tolist()

In [96]:
pc = productivity['Area'].unique().tolist()

In [97]:
for c in yc:
    if c not in pc:
        print(c)


In [98]:
c = set(pc + yc)

In [99]:
len(c)

212

# Fusion des datasets

In [100]:
# Fusion des datasets
dataset = productivity.merge(average_precipitation_per_year, how='left', on=['Area', 'Year'])
dataset = dataset.merge(pesticides, how='left', on=['Area', 'Year'])
dataset = dataset.merge(temp, how='left', on=['Area', 'Year'])
dataset.head(5)

,Area Code,Area,Item,Year,hg/ha,Code,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,2,Afghanistan,Maize,1961,14000,AFG,375.61630,NaN,14.23
1,2,Afghanistan,Maize,1962,14000,AFG,297.93750,NaN,14.10
2,2,Afghanistan,Maize,1963,14260,AFG,340.40457,NaN,15.01
3,2,Afghanistan,Maize,1964,14257,AFG,381.36438,NaN,13.73
4,2,Afghanistan,Maize,1965,14400,AFG,387.33570,NaN,13.90


In [101]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 56717 entries, 0 to 56716
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Area Code                      56717 non-null  int64  
 1   Area                           56717 non-null  str    
 2   Item                           56717 non-null  str    
 3   Year                           56717 non-null  int64  
 4   hg/ha                          56717 non-null  int64  
 5   Code                           44414 non-null  str    
 6   average_rain_fall_mm_per_year  44414 non-null  float64
 7   pesticides_tonnes              24153 non-null  float64
 8   avg_temp                       32210 non-null  float64
dtypes: float64(3), int64(3), str(3)
memory usage: 5.0 MB


On remarque que nous avons beaucoup de valeurs manquantes pour les précipitations, les pesticides et les températures. On va donc chercher à compléter du mieux possible ces valeurs manquantes avec d'autres datasets issues d'internet et de sources fiables.

Néanmoins, pour être sûr de fusionner correctement les datasets, il faut être sûr que les Area sont orthographiquement identiques entre les datasets.
Plutôt que de se baser sur les Area, on va se baser sur les Area Code qui sont des codes uniques de chaque pays, mais ces Area Code ne correspondent pas à des codes ISO.

Heureusement, j'ai retrouvé la correspondance entre les Area Code et les codes ISO sur le site de la FAO https://www.fao.org/home/en/. On va donc créer un dataset contenant cette correspondance.

# Iso Code et Area Code

In [102]:
Production_Crops_Livestock_E_AreaCodes = pd.read_csv('../data/raw/Production_Crops_Livestock_E_AreaCodes.csv', sep=',', na_values=[''], quotechar='"')
Production_Crops_Livestock_E_AreaCodes.head(5)

,Area Code,M49 Code,Area
0,2,4,Afghanistan
1,5100,2,Africa
2,3,8,Albania
3,4,12,Algeria
4,5200,19,Americas


In [103]:
a = productivity.copy()
a = a.merge(Production_Crops_Livestock_E_AreaCodes[['M49 Code', 'Area Code']], how='left', on='Area Code')
a['M49 Code'] = a['M49 Code'].fillna(0).astype('int64')
a.rename(columns={'M49 Code': 'ISO Code'}, inplace=True)

In [104]:
a.head(5)

,Area Code,Area,Item,Year,hg/ha,ISO Code
0,2,Afghanistan,Maize,1961,14000,4
1,2,Afghanistan,Maize,1962,14000,4
2,2,Afghanistan,Maize,1963,14260,4
3,2,Afghanistan,Maize,1964,14257,4
4,2,Afghanistan,Maize,1965,14400,4


In [105]:
# Est ce que tous les Area Code ont un code ISO correspondant ?
empty_m49_code = a[a['ISO Code'] == 0]
empty_m49_code['Area'].unique()

<ArrowStringArray>
[                 'American Samoa',                         'Bermuda',
                  'Cayman Islands',                            'Guam',
                      'Montserrat', 'Pacific Islands Trust Territory',
       'Wallis and Futuna Islands']
Length: 7, dtype: str

In [106]:
last_missing_codes = {
    'American Samoa': 16,
    'Cayman Islands': 136,
    'Bermuda': 60,
    'Guam': 316,
    'Montserrat': 500,
    'Pacific Islands Trust Territory': 583,
    'Wallis and Futuna Islands': 876,
}

for area, code in last_missing_codes.items():
    a.loc[a['Area'] == area, 'ISO Code'] = code

In [107]:
# Est ce que tous les Area Code ont un code ISO correspondant ?
empty_m49_code = a[a['ISO Code'] == 0]
empty_m49_code['Area'].unique()

<ArrowStringArray>
[]
Length: 0, dtype: str

In [108]:
import pandas as pd
wikipedia_iso_country_codes = pd.read_csv('../data/raw/wikipedia-iso-country-codes.csv', sep=',')
wikipedia_iso_country_codes

,English short name lower case,Alpha-2 code,Alpha-3 code,Numeric code,ISO 3166-2
0,Afghanistan,AF,AFG,4,ISO 3166-2:AF
1,Åland Islands,AX,ALA,248,ISO 3166-2:AX
2,Albania,AL,ALB,8,ISO 3166-2:AL
3,Algeria,DZ,DZA,12,ISO 3166-2:DZ
4,American Samoa,AS,ASM,16,ISO 3166-2:AS
...,...,...,...,...,...
241,Wallis and Futuna,WF,WLF,876,ISO 3166-2:WF
242,Western Sahara,EH,ESH,732,ISO 3166-2:EH
243,Yemen,YE,YEM,887,ISO 3166-2:YE
244,Zambia,ZM,ZMB,894,ISO 3166-2:ZM


In [109]:
wikipedia_iso_country_codes.rename(columns={'Numeric code' : 'ISO Code'}, inplace=True)

In [110]:
b = a.copy()
b = b.merge(wikipedia_iso_country_codes[['Alpha-2 code', 'Alpha-3 code', 'ISO Code']], how='left', on='ISO Code')

In [111]:
b.head(5)

,Area Code,Area,Item,Year,hg/ha,ISO Code,Alpha-2 code,Alpha-3 code
0,2,Afghanistan,Maize,1961,14000,4,AF,AFG
1,2,Afghanistan,Maize,1962,14000,4,AF,AFG
2,2,Afghanistan,Maize,1963,14260,4,AF,AFG
3,2,Afghanistan,Maize,1964,14257,4,AF,AFG
4,2,Afghanistan,Maize,1965,14400,4,AF,AFG


In [112]:
b.drop(columns=['Area Code'], inplace=True)

In [113]:
b.rename(columns={'ISO Code' : 'Area ISO Code', 'Alpha-2 code' : 'Area ISO Alpha-2 Code', 'Alpha-3 code' : 'Area ISO Alpha-3 Code'}, inplace=True)

In [114]:
b.head(5)

,Area,Item,Year,hg/ha,Area ISO Code,Area ISO Alpha-2 Code,Area ISO Alpha-3 Code
0,Afghanistan,Maize,1961,14000,4,AF,AFG
1,Afghanistan,Maize,1962,14000,4,AF,AFG
2,Afghanistan,Maize,1963,14260,4,AF,AFG
3,Afghanistan,Maize,1964,14257,4,AF,AFG
4,Afghanistan,Maize,1965,14400,4,AF,AFG


In [115]:
if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

b.to_csv('../data/processed/yield.csv', index=False)